In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/train.csv').drop(columns='ID')
test = pd.read_csv('../data/test.csv').drop(columns='ID')
submission = pd.read_csv('../data/sample_submission.csv')

In [2]:
X_train = train.drop(columns='y')
y_train = train['y']

In [3]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
X_train_scaled = minmax.fit_transform(X_train)
X_test = minmax.transform(test)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train_scaled, y_train, test_size= 0.2, random_state = 33)

In [5]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

batch_size = 32 #TODO hyperparameter 어떻게 정하지

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor)

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle=True)

In [6]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE, self).__init__()
        
        # encoder network
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim*2)
        )
        
        # decoder network
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
        
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu+eps*std
    
    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = h.chunk(2, dim=-1)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decoder(z)
        return recon_x, mu, logvar

In [7]:
def loss_function(recon_x, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(recon_x,x,reduction='sum')
    kl_div = -0.5*torch.sum(1+logvar-mu.pow(2)-logvar.exp())
    
    return recon_loss + kl_div

In [10]:
def train_vae(model, train_loader, val_loader, epochs , lr):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    for epoch in range(epochs):
        train_loss = 0
        for batch_x, _ in train_loader:
            optimizer.zero_grad()
            recon_x, mu, logvar = model(batch_x)
            loss = loss_function(recon_x, batch_x, mu, logvar)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        val_loss = 0
        model.eval() #TODO : 이거 왜함
        with torch.no_grad():
            for batch_x, _ in val_loader:
                recon_x, mu, logvar = model(batch_x)
                loss = loss_function(recon_x, batch_x, mu, logvar)
                val_loss += loss.item()
        if epoch % 10 == 0:
            print(f'Epoch: {epoch}, Train Loss: {train_loss/len(train_loader.dataset):.4f}, Validation Loss: {val_loss/len(val_loader.dataset):.4f}')

In [11]:
input_dim = X_train.shape[1]
hidden_dim = 10
latent_dim = 7
vae = VAE(input_dim,hidden_dim, latent_dim)
train_vae(vae, train_loader, val_loader, 100, 1e-3)

Epoch: 0, Train Loss: 1.1310, Validation Loss: 0.4365
Epoch: 10, Train Loss: 0.4174, Validation Loss: 0.4083
Epoch: 20, Train Loss: 0.4175, Validation Loss: 0.4082
Epoch: 30, Train Loss: 0.4173, Validation Loss: 0.4071
Epoch: 40, Train Loss: 0.4171, Validation Loss: 0.4072
Epoch: 50, Train Loss: 0.4170, Validation Loss: 0.4073
Epoch: 60, Train Loss: 0.4171, Validation Loss: 0.4079
Epoch: 70, Train Loss: 0.4171, Validation Loss: 0.4074
Epoch: 80, Train Loss: 0.4171, Validation Loss: 0.4084
Epoch: 90, Train Loss: 0.4170, Validation Loss: 0.4073


In [14]:
def get_latent(model, data_loader, is_test= False):
    latent_vectors = []
    model.eval()
    with torch.no_grad():
        for batch in data_loader:
            
            if is_test:
                batch_x = batch[0]
            else:
                batch_x, _ = batch
                
            h = model.encoder(batch_x)
            mu, _ = h.chunk(2, dim=-1)
            latent_vectors.append(mu)
            
    return torch.cat(latent_vectors, dim=0)

train_latent = get_latent(vae, train_loader)
val_latent = get_latent(vae, val_loader)
test_latent = get_latent(vae, test_loader, is_test = True)

In [15]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error


def train_and_evaluate(model, model_name, train_latent, y_train, val_latent, y_val):
    model.fit(train_latent.numpy(),y_train)
    val_prediction = model.predict(val_latent.numpy())
    mse = mean_squared_error(y_val, val_prediction)
    print(f'{model_name} >> Validation MSE : {mse:.4f}')
    
    pred = model.predict(test_latent.numpy())
    submission['y'] = pred
    submission.to_csv(f'../results/result_{model}.csv', index = False)

    return mse

def train_models(train_latent, y_train, val_latent, y_val):
    results = {}
    
    linear = LinearRegression()
    results['Linear Regression'] = train_and_evaluate(linear, 'Linear Regression', train_latent, y_train, val_latent, y_val)
    
    ridge = Ridge(alpha = 1.0, random_state= 33)
    results['Ridge Regression'] = train_and_evaluate(ridge, 'Ridge Regression', train_latent, y_train, val_latent, y_val)
    
    lasso = Lasso(alpha=0.1, random_state= 33)
    results['Lasso Regression'] = train_and_evaluate(lasso, 'Lasso Regression', train_latent, y_train, val_latent, y_val)
    
    return results

In [16]:
train_models(train_latent, y_train, val_latent, y_val)

Linear Regression >> Validation MSE : 7.0251
Ridge Regression >> Validation MSE : 6.8239
Lasso Regression >> Validation MSE : 6.8239


{'Linear Regression': 7.025062456441376,
 'Ridge Regression': 6.823934747729243,
 'Lasso Regression': 6.823934747729243}